# Exercise: Environmental Compliance Checker with RAG

## Learning Objectives

In this exercise, you will:
1. Understand how Retrieval-Augmented Generation (RAG) works
2. Learn document chunking and vector embeddings
3. Implement semantic similarity search
4. Build a compliance system that returns structured JSON responses
5. See how RAG improves accuracy by grounding LLM responses in source documents

## Business Context

Manufacturing facilities must ensure their emissions comply with Clean Air Act regulations. This exercise demonstrates how RAG can automate compliance checking by:
- Taking facility pollution data (pollutant type, quantity, source)
- Searching relevant Clean Air Act sections using vector similarity
- Returning a structured compliance assessment with citations

This approach provides auditable, accurate compliance decisions grounded in actual regulatory text.

## Setup: Install Required Libraries

In [ ]:
# Install required packages
!pip3 install -q chromadb langchain langchain-community langchain-text-splitters pypdf sentence-transformers python-dotenv requests

In [ ]:
import os
import json
import requests
import chromadb
from pathlib import Path
from dotenv import load_dotenv
from chromadb.config import Settings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

# Load environment variables from llm-apis directory
env_path = Path('..') / '.env'
load_dotenv(dotenv_path=env_path)

print("✓ Libraries imported successfully")

## Understanding RAG (Retrieval-Augmented Generation)

**The Problem:** LLMs have knowledge cutoff dates and can hallucinate. For regulatory compliance, we need current, accurate information with citations.

**The Solution:** RAG enhances LLMs by:
1. **Chunking**: Breaking the Clean Air Act PDF into searchable segments
2. **Embedding**: Converting text to numerical vectors that capture meaning
3. **Vector Search**: Finding relevant regulatory sections using similarity
4. **Prompt Construction**: Providing retrieved context to the LLM
5. **Structured Output**: Getting JSON responses for systematic compliance checking

**Try experimenting with:**
- Different chunk sizes (affects retrieval precision)
- Number of chunks retrieved (affects context completeness)
- Different facility pollution scenarios

## Step 1: Load and Chunk the Clean Air Act

In [ ]:
# Load PDF
pdf_path = "clean_air_act.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"✓ Loaded {len(documents)} pages from Clean Air Act")

# Chunk the document
# TRY EXPERIMENTING: Change chunk_size to 500 or 2000 to see impact on retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"✓ Created {len(chunks)} chunks")

## Step 2: Generate Embeddings and Store in Vector Database

In [ ]:
# Initialize embedding model (converts text to 384-dimensional vectors)
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✓ Model loaded")

# Initialize ChromaDB with persistent storage
# Use absolute path to ensure web-demo can access it
from pathlib import Path
chroma_db_path = str(Path.cwd() / "chroma_db")
print(f"ChromaDB path: {chroma_db_path}")

chroma_client = chromadb.PersistentClient(path=chroma_db_path)

# Create collection (delete if exists for clean run)
try:
    chroma_client.delete_collection("clean_air_act")
    print("Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(name="clean_air_act")
print("✓ ChromaDB collection created and will persist to disk")

In [ ]:
# Embed and store all chunks
print("Generating embeddings and storing in vector database...")

for i in range(0, len(chunks), 100):
    batch = chunks[i:i+100]
    texts = [chunk.page_content for chunk in batch]
    metadatas = [chunk.metadata for chunk in batch]
    ids = [f"chunk_{j}" for j in range(i, i+len(batch))]
    embeddings = embedding_model.encode(texts).tolist()
    
    collection.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=ids)

print(f"✓ Stored {collection.count()} chunks")

## Step 3: Semantic Similarity Search

When a user submits facility pollution data, we search for the most relevant Clean Air Act sections.

In [ ]:
def similarity_search(query, n_results=3):
    """Search for relevant Clean Air Act sections"""
    query_embedding = embedding_model.encode([query]).tolist()
    return collection.query(query_embeddings=query_embedding, n_results=n_results)

# Test it
test_results = similarity_search("sulfur dioxide emission limits", n_results=2)
print("Test search results:")
print("="*80)
for i, (doc, meta) in enumerate(zip(test_results['documents'][0], test_results['metadatas'][0]), 1):
    print(f"\n[Chunk {i}] Page {meta.get('page', 'N/A')}")
    print("-"*80)
    print(doc)  # Show full chunk, not truncated
    print("-"*80)

## Step 4: AWS Bedrock Integration - Compliance Checker

Now we integrate AWS Bedrock to analyze compliance. The system will:
1. Take facility pollution data
2. Retrieve relevant regulatory sections
3. Return structured JSON with compliance status and reasoning

**Note:** This uses the Bedrock API key from `/Users/troymarrotte/Development/bis-438-genai/llm-apis/.env`

In [ ]:
# Initialize Bedrock API configuration
bedrock_api_key = os.getenv('BEDROCK_API_KEY')
aws_region = os.getenv('AWS_REGION', 'us-east-1')

# Bedrock API endpoint
bedrock_endpoint = f"https://bedrock-runtime.{aws_region}.amazonaws.com/model/anthropic.claude-3-haiku-20240307-v1:0/invoke"

# Check if API key is configured
if not bedrock_api_key:
    print("❌ BEDROCK_API_KEY not found in .env file")
    print("\nExpected file: /Users/troymarrotte/Development/bis-438-genai/llm-apis/.env")
else:
    print(f"✓ Bedrock API configured ({aws_region})")
    print(f"✓ API Key: {bedrock_api_key[:20]}...")

In [ ]:
def check_compliance(facility_data, n_results=3):
    """
    Check if facility emissions comply with Clean Air Act.
    Returns structured JSON with compliance status and reasoning.
    """
    # Retrieve relevant sections
    search_results = similarity_search(facility_data, n_results)
    
    # Format context
    context_parts = []
    sources = []
    for i, (doc, metadata) in enumerate(zip(search_results['documents'][0], search_results['metadatas'][0]), 1):
        page = metadata.get('page', 'N/A')
        context_parts.append(f"[Source {i} - Page {page}]\n{doc}")
        sources.append(f"Page {page}")
    
    context = "\n\n".join(context_parts)
    
    # Construct prompt
    system_prompt = """You are an environmental compliance expert. Analyze facility emissions data against Clean Air Act requirements.

RESPOND ONLY WITH VALID JSON in this format:
{
  "compliance_status": "COMPLIANT" | "NON-COMPLIANT" | "INSUFFICIENT_INFO",
  "reasoning": "Brief explanation citing specific regulations",
  "relevant_sections": ["List regulatory sections that apply"],
  "recommendation": "What action is required or confirmation of compliance"
}

Base your assessment ONLY on the provided Clean Air Act context. If you do not have adequate information to determine compliance, respond with "INSUFFICIENT_INFO" and explain what data is missing."""
    
    user_prompt = f"""Clean Air Act Context:
{context}

---

Facility Emission Data:
{facility_data}

---

Provide compliance assessment as JSON."""
    
    # Call Bedrock via HTTP API
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "temperature": 0.2,
        "messages": [{"role": "user", "content": user_prompt}],
        "system": system_prompt
    }
    
    headers = {
        "Authorization": f"Bearer {bedrock_api_key}",
        "Content-Type": "application/json"
    }
    
    response = requests.post(bedrock_endpoint, headers=headers, json=request_body)
    
    if response.status_code != 200:
        raise Exception(f"Bedrock API error: {response.status_code} - {response.text}")
    
    response_body = response.json()
    llm_output = response_body['content'][0]['text']
    
    # Extract JSON
    if '```json' in llm_output:
        llm_output = llm_output.split('```json')[1].split('```')[0].strip()
    elif '```' in llm_output:
        llm_output = llm_output.split('```')[1].split('```')[0].strip()
    
    result = json.loads(llm_output)
    result['sources_used'] = sources
    
    return result

print("✓ Compliance checker ready")

## Test Compliance Scenarios

Let's test with different facility pollution profiles.

In [ ]:
# Scenario 1: Sulfur Dioxide Emissions - New Source Performance Standards
facility_1 = """Facility: Acme Steel Mill, Cleveland, Ohio
Pollutant: Sulfur Dioxide (SO2)
Annual Emissions: 95 tons/year
Source: Coal-fired boiler, 250 million BTU/hour capacity
Construction Date: 2015

Question: Does this facility qualify as a new source under Section 111? What are the standards of performance and emission limits that apply to our coal-fired boiler?"""

print("SCENARIO 1: SO2 Emissions\n" + "="*60)
result = check_compliance(facility_1)
print(f"Status: {result['compliance_status']}")
print(f"\nReasoning: {result['reasoning']}")
print(f"\nRelevant Sections: {', '.join(result['relevant_sections'])}")
print(f"\nRecommendation: {result['recommendation']}")
print(f"\nSources: {', '.join(result['sources_used'])}")

In [ ]:
# Scenario 2: Hazardous Air Pollutant - Section 112 Requirements
facility_2 = """Facility: ChemCorp Processing Plant, Houston, Texas
Pollutant: Benzene (Hazardous Air Pollutant)
Annual Emissions: 12 tons/year
Source: Chemical processing unit
Control Technology: Maximum Achievable Control Technology (MACT) installed

Question: Under Section 112 for hazardous air pollutants, what are the requirements for benzene emissions? Are we considered a major source and what emission standards apply?"""

print("\nSCENARIO 2: Benzene Emissions\n" + "="*60)
result = check_compliance(facility_2)
print(f"Status: {result['compliance_status']}")
print(f"\nReasoning: {result['reasoning']}")
print(f"\nRecommendation: {result['recommendation']}")
print(f"\nSources: {', '.join(result['sources_used'])}")

In [ ]:
# Scenario 3: New Source Construction - Prevention of Significant Deterioration
facility_3 = """Proposed Facility: New cement manufacturing plant, Phoenix, Arizona
Pollutants:
- Nitrogen Oxides (NOx): 180 tons/year
- Particulate Matter (PM): 250 tons/year
Status: Construction planned, not yet built

Question: What preconstruction permits and Prevention of Significant Deterioration (PSD) requirements apply under Part C? Do we need a Title V operating permit?"""

print("\nSCENARIO 3: New Construction\n" + "="*60)
result = check_compliance(facility_3)
print(f"Status: {result['compliance_status']}")
print(f"\nReasoning: {result['reasoning']}")
print(f"\nRecommendation: {result['recommendation']}")
print(f"\nSources: {', '.join(result['sources_used'])}")

In [ ]:
# Scenario 4: NON-COMPLIANT — Major HAP Source with No Controls and No Permits
# This facility clearly violates the Clean Air Act on multiple grounds:
#   1. Aggregate HAP emissions (80 tons/year) far exceed the 25 ton/year major source threshold (Section 112)
#   2. No Maximum Achievable Control Technology (MACT) installed despite being a major source
#   3. No Title V operating permit and no preconstruction permit ever obtained
facility_4 = """Facility: Riverside Chemical Plant, Gary, Indiana
Operations: Chemical manufacturing and waste solvent incineration
Pollutants Emitted:
- Benzene: 25 tons/year (Hazardous Air Pollutant)
- Vinyl Chloride: 15 tons/year (Hazardous Air Pollutant)
- Hydrogen Chloride: 40 tons/year
Aggregate HAP Emissions: 80 tons/year
Control Technology: NONE — no emission control equipment of any kind installed
Permit Status: No Title V operating permit, no preconstruction permit ever obtained
Construction Date: 2018

The facility has been operating continuously since 2018 without installing any air pollution control devices. We have never applied for or obtained any operating or construction permits. Our aggregate hazardous air pollutant emissions clearly exceed the major source thresholds.

Question: Are we in compliance with the Clean Air Act?"""

print("SCENARIO 4: Non-Compliant — Major HAP Source, No Controls, No Permits\n" + "="*60)
result = check_compliance(facility_4)
print(f"Status: {result['compliance_status']}")
print(f"\nReasoning: {result['reasoning']}")
print(f"\nRelevant Sections: {', '.join(result['relevant_sections'])}")
print(f"\nRecommendation: {result['recommendation']}")
print(f"\nSources: {', '.join(result['sources_used'])}")

## Web Demo

A Flask web application is available in the `web-demo/` subdirectory. The interface provides:

- Interactive form for entering facility pollution data
- Real-time compliance analysis
- Display of retrieved document chunks with similarity scores
- **Sample query buttons** for quick testing (SO2, Benzene, New Source)
- Visual compliance status

**Before starting the web server**, verify your vector database is ready by running the cell below.

Run the cell below to start the web server, then open http://localhost:5002 in your browser.

**IMPORTANT: Click the STOP button (■) when finished.**

In [ ]:
# Verify ChromaDB is ready for web demo
from pathlib import Path

chroma_db_path = Path.cwd() / "chroma_db"
if not chroma_db_path.exists():
    print("❌ ERROR: ChromaDB not found!")
    print("\nYou must run cells 6-10 first to create the vector database.")
    print("Go back and run:")
    print("  • Cell 7 (Load PDF)")
    print("  • Cell 9 (Initialize ChromaDB)")
    print("  • Cell 10 (Generate embeddings)")
else:
    # Verify collection exists
    test_client = chromadb.PersistentClient(path=str(chroma_db_path))
    try:
        test_collection = test_client.get_collection("clean_air_act")
        count = test_collection.count()
        print(f"✓ ChromaDB ready: {count} chunks stored")
        print(f"✓ Database location: {chroma_db_path}")
        print("\n🚀 Ready to start web server!")
    except Exception as e:
        print(f"❌ ERROR: Collection not found: {e}")
        print("\nRun cells 6-10 to create the database.")

In [ ]:
# Start the Flask web server
import subprocess
import sys
import os

# Install web-demo dependencies first
print("Installing web demo dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "web-demo/requirements.txt"], check=True)

os.chdir('web-demo')

print("\n" + "="*80)
print("Starting Flask web server...")
print("="*80)
print("\nServer: http://localhost:5002")
print("\nTo access:")
print("  • Local: Open http://localhost:5002 in browser")
print("  • Codespaces: Click popup to open forwarded port")
print("\n✓ Sample queries are available in the web interface")
print("\nPress STOP button (■) above to shut down when done.")
print("="*80)
print()

subprocess.run([sys.executable, "app.py"])

## Reflection Questions

1. How might the quality of the source documents used in a RAG system affect the trustworthiness of the answers it generates, particularly in a compliance context?

2. What are the potential risks of relying solely on a language model's retrieved output when making compliance decisions, and how might those risks be mitigated in a real-world setting?

3. In what ways do you think a RAG-based approach is better or worse suited for compliance use cases compared to a human expert reviewing the same documents?